# Churn Model — Logistic Regression on `ml_customer_features`

Trains a baseline binary classifier on the customer-level features produced by `ml_features.ipynb`. Output: a fitted sklearn `LogisticRegression` and a held-out AUC.

**Why sklearn on a sample, not Spark ML:** Databricks serverless and shared-mode clusters whitelist Py4J constructors for security. `pyspark.ml.feature.VectorAssembler` (and most of `pyspark.ml`) is **not** whitelisted, so any Spark ML estimator instantiation raises `Py4JSecurityException`. The workaround is to keep all heavy aggregation in Spark (already done in `ml_features.ipynb`), then sample down to a pandas DataFrame for the actual modeling step. For a baseline LR on ~tens of features, a 1M-row sample is more than enough — the training set is small enough that sklearn fits in seconds and the AUC estimate is tight.

**Pipeline:**
1. Load `ml_customer_features` via `spark.table()`.
2. Sample to a tractable size and convert to pandas.
3. 80/20 split + sklearn `LogisticRegression`.
4. Evaluate with `roc_auc_score` and `average_precision_score`.

**Design choices:**

- **Numeric features only for the baseline.** `loyalty_tier` and `primary_region` are categorical — they'd need one-hot encoding to enter a linear model. Worth adding once the numeric-only baseline is validated.
- **`customer_id` is excluded.** It's an identifier, not a feature; including it would let the model memorize.
- **Date columns excluded.** Their information is already encoded in `customer_tenure_days` / `days_since_last_purchase`. Keeping both injects collinearity.
- **`StandardScaler` before LR.** sklearn's LR doesn't standardize internally (Spark ML does); without scaling, the L-BFGS solver converges slowly and `regParam` becomes meaningless across features of different magnitudes.
- **Sampling fraction.** 0.005 of the source table at 200M customers is ~1M rows — fast and representative. Increase to 0.05 if you want tighter AUC estimates; the only cost is `toPandas()` time.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.pipeline import Pipeline

spark = SparkSession.builder.getOrCreate()

## 1. Load + sample to pandas (with on-the-fly label recomputation)

Read the persisted feature table written by `ml_features.ipynb`, but **override `churn_label` here** rather than trust the persisted version. Reason: the persisted label uses a hard 60-day cutoff, which collapses to a single class on datasets where every customer has recent activity (e.g. 100M transactions ÷ 100K customers = ~1000 txns each, so everyone's last purchase is in the last week).

**First attempt (didn't work):** define churn as `days_since_last_purchase > P75`. The intent was a self-calibrating ~25% positive class, but with this dataset's density (~1000 txns/customer/year) `days_since_last_purchase` is degenerate — the value is 0 for the overwhelming majority. `percentile_approx(..., 0.75)` then returns 0, and `> 0` is true for almost no rows. Worse, after sampling at 0.005 the few non-zero rows usually don't survive, so the sample comes back single-class and `LogisticRegression.fit` raises `ValueError: ... data contains only one class: 0`.

**Fix:** use a **rank-based** label instead of a value-based cutoff. Rank the sampled rows by `days_since_last_purchase` and label the top quartile as churned. Rank ordering produces a clean 25/75 split regardless of how concentrated the underlying values are — ties are broken by position via `method="first"`. We do this in pandas after sampling (cheap on ~1M rows) rather than in Spark, since `percentile_approx` on a degenerate distribution can't help us here.

`spark.sample(fraction, seed)` runs entirely in Spark — only the sampled rows are shipped to the driver via `toPandas()`. Driver memory bound is `sample_fraction × source_rows × bytes_per_row`, so size the fraction to fit comfortably in driver heap.

In [ ]:
SAMPLE_FRACTION = 0.005  # ~1M rows out of 200M; tune up for tighter AUC
CHURN_QUANTILE = 0.75    # top (1 - q) by recency are labeled churned → 25% positive class

# Numeric feature list. Maintain explicitly (not "all numeric columns") so accidental
# inclusion of a leaky column when the source schema grows is caught.
feature_cols = [
    "total_gross_revenue",
    "total_discount",
    "total_revenue",
    "total_cost",
    "total_gross_margin",
    "total_quantity",
    "transaction_count",
    "promoted_transaction_count",
    "avg_discount_pct",
    "distinct_categories",
    "distinct_brands",
    "distinct_stores",
    "distinct_promotion_types",
    "avg_transaction_value",
    "gross_margin_pct",
    "promo_transaction_pct",
    "customer_tenure_days",
    "days_since_last_purchase",
]
target_col = "churn_label"

# Load + project. Drop NULLs on the recency column since it drives the label override.
ml_data = (
    spark.table("ml_customer_features")
    .select(*feature_cols)
    .dropna(subset=feature_cols)
)

# --- Sample to pandas FIRST -----------------------------------------------------------
# We label after sampling because the underlying distribution of days_since_last_purchase
# is degenerate (mostly 0) — a value-based cutoff via percentile_approx collapses to a
# single class. A rank-based label is the right tool when values are concentrated.
sample_pdf = (
    ml_data
    .sample(fraction=SAMPLE_FRACTION, seed=42)
    .toPandas()
)
print(f"Sampled rows: {len(sample_pdf):,}")

# --- Label recomputation (rank-based) -------------------------------------------------
# rank(pct=True) gives the percentile rank in [0, 1]. method="first" breaks ties by row
# order so identical values get distinct ranks — critical here, because most rows have
# days_since_last_purchase == 0 and we still need a clean 25/75 split.
recency_pct_rank = sample_pdf["days_since_last_purchase"].rank(method="first", pct=True)
sample_pdf[target_col] = (recency_pct_rank > CHURN_QUANTILE).astype(int)

print(f"Class balance:\n{sample_pdf[target_col].value_counts().to_string()}")
print(f"Positive rate: {sample_pdf[target_col].mean():.4f}")

# Guard: if the sample still has only one class, train_test_split + LR will fail. Surface
# that early with a clear message rather than letting sklearn raise its generic ValueError.
if sample_pdf[target_col].nunique() < 2:
    raise ValueError(
        f"Sample has a single class for {target_col} after rank-based labeling — "
        f"this should be impossible unless the sample has <4 rows."
    )

## 2. Train/test split

`train_test_split` is the sklearn equivalent of Spark's `randomSplit`. `random_state=42` mirrors the seed we used in the Spark sample for reproducibility.

`stratify=y` ensures the train/test splits have the same churn rate. At a balanced dataset this is cosmetic, but at imbalanced rates (e.g. 5% churn) it prevents the test set from accidentally over- or under-representing the positive class.

In [ ]:
X = sample_pdf[feature_cols]
y = sample_pdf[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"Train rows: {len(X_train):,}")
print(f"Test rows:  {len(X_test):,}")
print(f"Train churn rate: {y_train.mean():.4f}")
print(f"Test churn rate:  {y_test.mean():.4f}")

## 3. Scale + train

sklearn's `LogisticRegression` does NOT standardize features internally (Spark ML does, which is why the Spark version of this notebook didn't need a scaler stage). Without standardization:
- L-BFGS converges much more slowly because the loss surface is poorly conditioned.
- `C` (sklearn's inverse regularization) becomes incomparable across features of different magnitudes.

We use a `Pipeline` so `StandardScaler` is fit on training data only — its mean/std come from `X_train` and are applied identically to `X_test` at evaluation. Manually scaling with `fit_transform(X_train)` then `transform(X_test)` works too, but the Pipeline eliminates the chance of accidentally calling `fit_transform` on the test set.

`max_iter=1000` is generous; LR usually converges in <100. `solver="lbfgs"` is the default and matches what Spark ML uses, making numerical comparisons between the two implementations meaningful.

In [ ]:
model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(
        max_iter=1000,
        solver="lbfgs",
        C=1.0,           # default; lower = more regularization
        n_jobs=-1,       # use all driver cores for the LR fit
    )),
])

model.fit(X_train, y_train)

# Convergence diagnostic — if n_iter equals max_iter, increase max_iter or scale issues
# weren't actually fixed.
lr_step = model.named_steps["lr"]
print(f"L-BFGS iterations: {lr_step.n_iter_[0]} (cap: {lr_step.max_iter})")
print(f"Training AUC:      {roc_auc_score(y_train, model.predict_proba(X_train)[:, 1]):.4f}")

## 4. Evaluate on the held-out test set

AUC-ROC is the right primary metric for this problem:
- **Threshold-independent.** Unlike accuracy/precision/recall, it doesn't depend on where we set the decision boundary. For churn we'll usually set the threshold by business cost (cost of intervention vs cost of letting a customer churn), not by 0.5.
- **Robust to class imbalance.** If churn rate is 5%, a constant-zero classifier has 95% accuracy but 0.5 AUC. The latter is the honest signal.

We also compute **average precision** (PR-AUC). For imbalanced classes it's more discriminating than ROC AUC at the high-precision end of the curve — which is the operating regime that actually matters for churn intervention.

**Sanity-check coefficients** afterward: a baseline LR should show negative coefficients on `total_revenue`, `transaction_count`, `customer_tenure_days` (more engagement → less churn) and a positive coefficient on `days_since_last_purchase` (the label is *defined* from this column, so the relationship has to be strong and positive). If signs come out wrong, the model has a bug — not a modeling problem.

In [ ]:
# predict_proba returns shape (n, 2): column 0 = P(churn=0), column 1 = P(churn=1).
# AUC takes the positive-class probability.
test_proba = model.predict_proba(X_test)[:, 1]

test_auc = roc_auc_score(y_test, test_proba)
test_pr_auc = average_precision_score(y_test, test_proba)
print(f"Test AUC-ROC: {test_auc:.4f}")
print(f"Test AUC-PR:  {test_pr_auc:.4f}")

# Coefficient sanity check. Note: scaler standardized the features, so coefficients here
# are on the STANDARDIZED scale — directly comparable in magnitude across features (a
# coefficient of 0.5 means "+0.5 logit per +1 standard deviation of this feature").
# This is actually more interpretable than raw-scale coefficients for ranking importance.
print("\nFeature coefficients (standardized, sorted by absolute value):")
coeffs = list(zip(feature_cols, lr_step.coef_[0]))
for name, c in sorted(coeffs, key=lambda x: abs(x[1]), reverse=True):
    print(f"  {name:<30s} {c:+.4f}")
print(f"  {'(intercept)':<30s} {lr_step.intercept_[0]:+.4f}")

# Sample predictions for visual sanity check.
import pandas as pd
preview = pd.DataFrame({
    "churn_label": y_test.values[:10],
    "predicted_label": model.predict(X_test)[:10],
    "churn_probability": test_proba[:10].round(4),
})
print("\nSample test predictions:")
print(preview.to_string(index=False))